In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.backends.cudnn as cudnn
import torchvision
import torchvision.transforms as transforms
import os
import argparse
from pathlib import Path
import re
import random
import math
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
from torch.utils.data import Dataset, DataLoader
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Model and Loss

In [ ]:
"""GP-VAE PyTorch port adapted to MAP/ScO2 (B,1080,2).

Source: https://github.com/ratschlab/GP-VAE (Fortuin et al., AISTATS 2020)
References: lib/models.py (JointEncoder / BandedJointEncoder, GaussianDecoder,
GP_VAE / HI_VAE), lib/nn_utils.py, lib/gp_kernel.py, train.py.

Faithful source concepts:
  - CNN time-series encoder (one Conv1D followed by pointwise Dense layers)
  - either diagonal JointEncoder posterior (EXACT analytic Gaussian KL) or
    banded-precision BandedJointEncoder posterior (structured Gaussian)
  - Gaussian decoder with fixed scale=1, not MAE
  - Cauchy GP prior with 1e-3 diagonal jitter (source convention)
  - negative ELBO = observed-only Gaussian NLL summed per sequence
                    + beta * full-trajectory GP KL
  - sampled latent z in training; posterior mean for deterministic imputation

Differences from the released TensorFlow model:
  - PyTorch, 1080 samples and 2 channels; explicit observed mask
  - optional normalization (the source takes externally preprocessed data)
  - banded mode computes an unbiased *one-sample* Monte-Carlo KL estimate,
    rather than the source's analytic KL (avoids O(B * Z * T**2) covariance
    and O(T**3) per-posterior operations at T=1080). For the exact source
    analytic ELBO use posterior='diagonal' or original TF code.
  - forward returns the dict keys used by the user's existing training code.

IMPORTANT: Hidden ground-truth values and I_mask are NEVER used in ELBO.
Evaluation MAE/MAPE on I_mask is separate. Any former GP-VAE MAE checkpoints
are architecturally incompatible; retrain using beta=0.2 (PhysioNet setting).
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class GPVAEEncoder(nn.Module):
    """Official JointEncoder / BandedJointEncoder style."""
    def __init__(self, data_dim=2, latent_dim=35, hidden_sizes=(128, 128),
                 window_size=24, posterior='diagonal', data_type='physionet',
                 **_unused):
        super().__init__()
        if not hidden_sizes:
            raise ValueError('hidden_sizes must contain at least one value')
        if posterior not in ('diagonal', 'banded'):
            raise ValueError("posterior must be 'diagonal' or 'banded'")
        self.latent_dim = int(latent_dim)
        self.posterior = posterior
        self.data_type = data_type
        self.conv = nn.Conv1d(data_dim, hidden_sizes[0], kernel_size=window_size,
                              padding='same')
        dims = list(hidden_sizes)
        self.hidden = nn.ModuleList(nn.Linear(a, b) for a, b in zip(dims[:-1], dims[1:]))
        self.to_params = nn.Linear(dims[-1], (3 if posterior == 'banded' else 2)*latent_dim)

    def forward(self, observed_normalized, m_hat=None):
        # Original network receives missing values as zeros, not a concatenated mask.
        h = self.conv(observed_normalized.transpose(1, 2)).transpose(1, 2)
        for layer in self.hidden:
            h = F.relu(layer(h))
        params = self.to_params(h)
        mu = params[..., :self.latent_dim]   # B,T,Z
        if self.posterior == 'diagonal':
            sd = F.softplus(params[..., self.latent_dim:]) + 1e-5
            return mu, {'std': sd, 'logvar': 2.0*torch.log(sd)}
        # Published banded encoder produces one positive diagonal and one
        # positive superdiagonal of an upper precision Cholesky factor.
        raw = params[..., self.latent_dim:]
        transform = torch.sigmoid if self.data_type == 'physionet' else F.softplus
        diag = 1.0 + transform(raw[..., :self.latent_dim])
        upper = transform(raw[..., self.latent_dim:])[:, :-1, :]
        return mu, {'diag': diag, 'upper': upper}


class GPVAEDecoder(nn.Module):
    """Source GaussianDecoder: dense ReLU layers + unit-variance Normal."""
    def __init__(self, latent_dim=35, data_dim=2, hidden_sizes=(256, 256), **_unused):
        super().__init__()
        dims = [latent_dim] + list(hidden_sizes) + [data_dim]
        layers = []
        for a, b in zip(dims[:-2], dims[1:-1]):
            layers.extend([nn.Linear(a, b), nn.ReLU()])
        layers.append(nn.Linear(dims[-2], dims[-1]))
        self.net = nn.Sequential(*layers)

    def forward(self, z):
        return self.net(z)


def _cauchy_prior_matrix(T, sigma, length_scale, kernel, jitter):
    t = torch.arange(T, dtype=torch.float64)
    squared_dist = (t[:, None]-t[None, :]).square()
    if kernel == 'cauchy':
        k = sigma / (1.0 + squared_dist/(length_scale**2))
    elif kernel == 'rbf':
        k = torch.exp(-squared_dist/(length_scale**2))
    elif kernel == 'matern':
        k = torch.exp(-torch.sqrt(squared_dist)/math.sqrt(length_scale))
    elif kernel == 'diffusion':
        if not 0 < length_scale < 0.5:
            raise ValueError('diffusion kernel requires 0 < length_scale < 0.5')
        k = torch.eye(T, dtype=torch.float64)*(1.0-length_scale)
        k += torch.diag(torch.full((T-1,), length_scale, dtype=torch.float64), 1)
        k += torch.diag(torch.full((T-1,), length_scale, dtype=torch.float64), -1)
    else:
        raise ValueError(f'Unknown kernel: {kernel}')
    if kernel == 'cauchy':
        k += torch.eye(T, dtype=torch.float64)*jitter
    else:
        # The source does not add jitter to other kernels; add only enough
        # numerical jitter when required, made explicit here.
        k += torch.eye(T, dtype=torch.float64)*jitter
    return k


class GPVAEImputer(nn.Module):
    def __init__(self, seq_len=1080, data_dim=2, latent_dim=35,
                 encoder_sizes=(128, 128), decoder_sizes=(256, 256),
                 window_size=24, kernel='cauchy', gp_length_scale=7.0,
                 gp_sigma=1.005, gp_jitter=0.001, kernel_scales=1,
                 beta=0.2, posterior='banded', data_type='physionet',
                 normalization='observed', eps=1e-6,
                 # Compatibility with earlier constructor:
                 hidden_dim=None, dropout=None, **kwargs):
        super().__init__()
        if kwargs:
            raise TypeError('Unexpected GPVAEImputer arguments: '+str(sorted(kwargs)))
        if hidden_dim is not None:
            # Earlier code used three 1d convolutions; official code uses one.
            # Here use hidden_dim for both CNN + dense hidden layer if requested.
            encoder_sizes = (hidden_dim, hidden_dim)
            decoder_sizes = (hidden_dim, hidden_dim)
        self.seq_len = int(seq_len)
        self.data_dim = int(data_dim)
        self.latent_dim = int(latent_dim)
        self.posterior = posterior
        self.beta = float(beta)
        self.normalization = normalization
        self.eps = float(eps)
        if normalization not in ('observed', 'fold', 'none'):
            raise ValueError("normalization must be 'observed', 'fold', or 'none'")
        if not 1 <= kernel_scales <= latent_dim:
            raise ValueError('kernel_scales must be in [1,latent_dim]')
        self.encoder = GPVAEEncoder(data_dim, latent_dim, encoder_sizes, window_size,
                                     posterior=posterior, data_type=data_type)
        self.decoder = GPVAEDecoder(latent_dim, data_dim, decoder_sizes)
        self.register_buffer('fold_mu', torch.zeros(1, 1, data_dim))
        self.register_buffer('fold_sigma', torch.ones(1, 1, data_dim))
        self.register_buffer('fold_stats_ready', torch.tensor(False))
        precisions, diagonals, logdets = [], [], []
        for g in range(kernel_scales):
            length = gp_length_scale / 2**g
            K = _cauchy_prior_matrix(seq_len, gp_sigma, length, kernel, gp_jitter)
            L = torch.linalg.cholesky(K)
            inv = torch.cholesky_inverse(L)
            precisions.append(inv.float())
            diagonals.append(torch.diagonal(inv).float())
            logdets.append((2*torch.log(torch.diagonal(L)).sum()).float())
        self.register_buffer('gp_precision', torch.stack(precisions), persistent=False)
        self.register_buffer('gp_precision_diag', torch.stack(diagonals), persistent=False)
        self.register_buffer('gp_logdet', torch.stack(logdets), persistent=False)
        self.scale_ids = []
        n_assigned = 0
        for g in range(kernel_scales):
            n = latent_dim-n_assigned if g == kernel_scales-1 else math.ceil(latent_dim/kernel_scales)
            self.scale_ids.extend([g]*n)
            n_assigned += n
        assert len(self.scale_ids) == latent_dim

    @torch.no_grad()
    def set_standardization(self, mean, std):
        """Required only for normalization='fold'; call with TRAIN patients only."""
        mean = torch.as_tensor(mean, dtype=self.fold_mu.dtype, device=self.fold_mu.device)
        std = torch.as_tensor(std, dtype=self.fold_sigma.dtype, device=self.fold_sigma.device)
        self.fold_mu.copy_(mean.reshape(1, 1, self.data_dim))
        self.fold_sigma.copy_(std.reshape(1, 1, self.data_dim).clamp_min(self.eps))
        self.fold_stats_ready.fill_(True)

    def observed_normalize(self, x_in, m_hat):
        valid = (m_hat > 0.5) & torch.isfinite(x_in)
        observed = torch.where(valid, x_in, torch.zeros_like(x_in))
        if self.normalization == 'observed':
            n = valid.sum(dim=1, keepdim=True).clamp_min(1)
            mu = observed.sum(1, keepdim=True)/n
            sigma = (((observed-mu).square()*valid).sum(1, keepdim=True)/n+self.eps).sqrt()
            mu = torch.where(n > 0, mu, torch.zeros_like(mu))
        elif self.normalization == 'fold':
            if not bool(self.fold_stats_ready):
                raise RuntimeError("Call model.set_standardization(train_mean,train_std) for normalization='fold'.")
            mu, sigma = self.fold_mu, self.fold_sigma
        else:
            mu = torch.zeros(1, 1, self.data_dim, device=x_in.device, dtype=x_in.dtype)
            sigma = torch.ones_like(mu)
        xnorm = torch.where(valid, (observed-mu)/sigma, torch.zeros_like(observed))
        return xnorm, mu, sigma, valid

    def _kl_diagonal(self, z_mu, logvar):
        # Analytic KL(N(mu,diag(var)) || N(0,K_GP)) for all (B,Z,T).
        z = z_mu.transpose(1, 2)
        v = logvar.exp().transpose(1, 2)
        logv = logvar.transpose(1, 2)
        B,Z,T = z.shape
        kl_per_batch = z.new_zeros(B)
        for g in set(self.scale_ids):
            indices = [i for i, v_ in enumerate(self.scale_ids) if v_ == g]
            mu_g, var_g, lv_g = z[:, indices], v[:, indices], logv[:, indices]
            p = self.gp_precision[g]
            quad = ((mu_g @ p)*mu_g).sum((-1,-2))
            trace = (var_g * self.gp_precision_diag[g]).sum((-1,-2))
            logdet = self.gp_logdet[g] * len(indices)
            kl_per_batch += 0.5*(quad+trace - len(indices)*T + logdet - lv_g.sum((-1,-2)))
        return kl_per_batch.mean()

    def _draw_banded(self, z_mu, diag, upper):
        # Source: U is upper bidiagonal, q scale_tril = (U^-1)^T,
        # hence q covariance = U^-T U^-1 (NOT U^-1 U^-T).
        # z - mu = U^-T eps; explicit forward substitution O(B Z T).
        eps = torch.randn_like(z_mu)
        values = [None]*self.seq_len
        values[0] = eps[:, 0, :]/diag[:, 0, :]
        for t in range(1, self.seq_len):
            values[t] = (eps[:, t, :] - upper[:, t-1, :]*values[t-1])/diag[:, t, :]
        z = z_mu + torch.stack(values, dim=1)
        return z, eps

    def _kl_banded_mc(self, z_mu, diag, upper, z, eps):
        B,T,Z = z_mu.shape
        logq = (-0.5*eps.square().sum((1,2)) + torch.log(diag).sum((1,2))
                -0.5*T*Z*math.log(2*math.pi))
        z_p = z.transpose(1, 2)
        logp = z.new_zeros(B)
        for g in set(self.scale_ids):
            indices = [i for i, v_ in enumerate(self.scale_ids) if v_ == g]
            z_g = z_p[:, indices]
            quadratic = ((z_g @ self.gp_precision[g])*z_g).sum((-1,-2))
            logp += -0.5*(quadratic + len(indices)*self.gp_logdet[g]
                          +len(indices)*T*math.log(2*math.pi))
        return (logq - logp).mean()

    def forward(self, x_in, m_hat, sample=None):
        if x_in.ndim != 3 or x_in.shape[1:] != (self.seq_len, self.data_dim):
            raise ValueError(f'Expected (B,{self.seq_len},{self.data_dim}), got {tuple(x_in.shape)}')
        if m_hat.shape != x_in.shape:
            raise ValueError('x_in and m_hat shapes do not match')
        # Use FP32 for the KL: dense GP inverses + autocast half can lose precision.
        with torch.autocast(device_type=x_in.device.type, enabled=False):
            x_norm, mu, sigma, obs = self.observed_normalize(x_in.float(), m_hat.float())
            z_mu, posterior_params = self.encoder(x_norm)
            sample = self.training if sample is None else bool(sample)
            if self.posterior == 'diagonal':
                z_logvar = posterior_params['logvar']
                z_sample = z_mu + torch.randn_like(z_mu)*posterior_params['std'] if sample else z_mu
                kl = self._kl_diagonal(z_mu, z_logvar)
            else:
                z_draw, eps = self._draw_banded(z_mu, posterior_params['diag'], posterior_params['upper'])
                z_sample = z_draw if sample else z_mu
                kl = self._kl_banded_mc(z_mu, posterior_params['diag'], posterior_params['upper'], z_draw, eps)
                diag, upper = posterior_params['diag'], posterior_params['upper']
                variances = [None]*self.seq_len
                variances[0] = diag[:, 0, :].square().reciprocal()
                for t in range(1, self.seq_len):
                    variances[t] = (1.0 + upper[:, t-1, :].square()*variances[t-1]) / diag[:, t, :].square()
                z_logvar = torch.stack(variances, dim=1).clamp_min(1e-30).log()
            x_norm_hat = self.decoder(z_sample)
            x_hat = x_norm_hat*sigma + mu
            x_c = torch.where(obs, x_in.float(), x_hat)
            return {'x_hat': x_hat, 'x_c': x_c, 'x_norm_hat': x_norm_hat,
                    'z_mu': z_mu, 'z_logvar': z_logvar, 'kl': kl,
                    'mu': mu, 'sigma': sigma, 'x_norm_in': x_norm,
                    'observed_mask': obs, 'beta': self.beta}


def gpvae_loss_same_I_mask(outputs, x_true, m_hat, I_mask,
                           lam_mit=None, beta_kl=None, loss_type=None):
    """Source-style negative GP-VAE ELBO, despite legacy compatibility name.

    Gaussian reconstruction NLL is evaluated ONLY at observed values; hidden
    x_true/I_mask are NEVER used to compute loss. The latter can optionally be
    used for logging withheld-point imputation MAE *after* prediction.

    Source computes nll SUM over time/channels per sample, then batch mean;
    KL is full trajectory Gaussian KL, not normalized by T or latent_dim.
    """
    if loss_type is not None and str(loss_type).lower() not in ('nll', 'elbo', 'gaussian'):
        raise ValueError("GP-VAE original objective is Gaussian NLL, not loss_type='mae'/'mse'. Remove loss_type.")
    if lam_mit is not None:
        raise ValueError('Original GP-VAE does not use lam_mit; remove it from the loss call.')
    obs = outputs['observed_mask']
    x0 = outputs['x_norm_in']
    phat = outputs['x_norm_hat']
    per_entry_nll = 0.5*((phat-x0).square() + math.log(2.0*math.pi))
    nll = torch.where(obs, per_entry_nll, torch.zeros_like(per_entry_nll)).sum((1,2)).mean()
    kl = outputs['kl']
    beta = float(outputs['beta'] if beta_kl is None else beta_kl)
    total = nll + beta*kl

    finite_gt = torch.isfinite(x_true)
    eval_mask = (I_mask > 0.5) & (~obs) & finite_gt
    safe_truth = torch.where(eval_mask, x_true, torch.zeros_like(x_true))
    safe_pred = torch.where(eval_mask, outputs['x_hat'], torch.zeros_like(outputs['x_hat']))
    mit_mae = (safe_pred-safe_truth).abs().sum()/eval_mask.sum().clamp_min(1)
    logs = {'loss': total.detach(), 'nll': nll.detach(), 'kl_loss': kl.detach(),
            'ort_loss': nll.detach(), 'mit_loss': mit_mae.detach(),
            'n_ort': obs.sum().detach(), 'n_mit': eval_mask.sum().detach(),
            'beta': torch.tensor(beta, device=total.device)}
    return total, logs

gpvae_original_elbo_loss = gpvae_loss_same_I_mask

### Model training, validation, and testing

In [ ]:
NPZ_PATH = (
    "/content/drive/MyDrive/aim3/"
    "saits_masks_1800x2_reintroduced_exact_gaps_capped360_with_bio_numeric.npz"
)
OUT_DIR = "/content/drive/MyDrive/aim3/GPVAE_OriginalELBO_Banded_5fold_1e-3_256"

In [ ]:
"""GP-VAE source-aligned negative-ELBO benchmark, 5-fold PATIENT-LEVEL CV.

RUN AFTER defining GPVAEImputer and gpvae_original_elbo_loss from:
    gpvae_original_objective_pytorch_1080x2.py

Model TRAINING objective (original GP-VAE):
    negative ELBO = observed Gaussian NLL + beta * full GP-prior KL.
    No artificially hidden values enter this training objective.

Checkpoint selection is an EXPLICIT task-specific ADAPTATION:
    validation MAE on artificially hidden points (true inference, no GT input).
    Validation ELBO/NLL/KL are also logged. Test GT is used only to measure
    held-out test MAE/RMSE/MAPE after the checkpoint has been selected.

This script defines NO model architecture. It preserves (B,1080,2), five-fold
patient splits, predictions, summaries, and the four corrected gap-duration bins.

The structured banded posterior and full-length 1080-point GP are expensive.
Use a small BATCH_SIZE and FP32; optionally choose posterior='diagonal' for
much faster execution, but document that posterior choice in your manuscript.
"""

import os
import random

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset

# ---------------------------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------------------------
CKPT_DIR = os.path.join(OUT_DIR, "ckpts")

SEED = 1337
N_SPLITS = 5
VAL_WITHIN_TRAIN_P = 0.10
SEQ_LEN = 1080
D_IN = 2

BATCH_SIZE = 256
GRAD_ACCUM_STEPS = 4
EPOCHS = 300
LR = 1e-3
WEIGHT_DECAY = 0.0 
EARLY_STOPPING_PATIENCE = 15
CLIP_GRAD_NORM = 1.0
NUM_WORKERS = 0
EPS = 1e-8

MODEL_KWARGS = dict(
    seq_len=SEQ_LEN,
    data_dim=D_IN,
    latent_dim=35,
    encoder_sizes=(128, 128),
    decoder_sizes=(256, 256),
    window_size=24,
    kernel="cauchy",
    gp_length_scale=7.0,
    gp_sigma=1.005,
    gp_jitter=1e-3,
    kernel_scales=1,
    beta=0.2,
    posterior="banded",  # Original structured posterior; 'diagonal' is faster
    normalization="observed",  # Patient/segment observed-only normalization
)

MODEL_NAME = "GPVAE_OriginalELBO_" + MODEL_KWARGS["posterior"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = DEVICE.type == "cuda"
REGIONS = ["<1 min", "[1,10) min", "[10,20) min", "[20,30] min"]


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class TSImputeDataset(Dataset):
    def __init__(self, x_in, m_hat, x_true, i_mask, seg_indices):
        self.x_in = torch.from_numpy(
            np.nan_to_num(x_in, nan=0., posinf=0., neginf=0.).astype(np.float32)
        )
        self.m_hat = torch.from_numpy(
            np.nan_to_num(m_hat, nan=0., posinf=0., neginf=0.).astype(np.float32)
        )
        self.x_true = torch.from_numpy(np.asarray(x_true, dtype=np.float32))
        self.i_mask = torch.from_numpy(
            np.nan_to_num(i_mask, nan=0., posinf=0., neginf=0.).astype(np.float32)
        )
        self.indices = torch.as_tensor(seg_indices, dtype=torch.long)
        assert self.x_in.shape == self.m_hat.shape == self.x_true.shape == self.i_mask.shape
        assert len(self.indices) == len(self.x_in)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        return (self.x_in[idx], self.m_hat[idx], self.x_true[idx],
                self.i_mask[idx], self.indices[idx])


def load_data():
    with np.load(NPZ_PATH, allow_pickle=True) as d:
        x_in = d['X_in']
        m_hat = d['M_hat']
        x_true = d['X']
        i_mask = d['I']
        seg_id = d['segment_ids']
        filenames = (np.asarray([str(v) for v in d['filenames']])
                     if 'filenames' in d.files else
                     np.array([f'segment_{i}' for i in range(len(seg_id))]))
        if 'target_columns' in d.files and len(d['target_columns']) == D_IN:
            file_names = list(map(str, d['target_columns']))
        elif 'columns_all' in d.files and len(d['columns_all']) == D_IN:
            file_names = list(map(str, d['columns_all']))
        else:
            file_names = None
    if x_in.ndim != 3 or x_in.shape[1:] != (SEQ_LEN, D_IN):
        raise ValueError(f'Expected (N,{SEQ_LEN},{D_IN}), got {x_in.shape}')
    if not (x_in.shape == m_hat.shape == x_true.shape == i_mask.shape):
        raise ValueError('NPZ signal and mask shapes disagree')
    if len(seg_id) != len(x_in):
        raise ValueError('segment_ids length mismatch')
    if np.any((m_hat > .5) & (i_mask > .5)):
        raise ValueError('M_hat and I overlap: target points must be withheld')
    if not np.isfinite(x_in[m_hat > .5]).all():
        raise ValueError('Nonfinite observed X_in values')
    # User specified this physical ordering, independent of NPZ metadata.
    target_names = ['ScO2', 'MAP']
    print('NPZ channel names:', file_names)
    print('Benchmark channel names (user-specified):', target_names)
    return x_in, m_hat, x_true, i_mask, seg_id, filenames, target_names


def make_loader(arrays, indices, train=False):
    x_in, m_hat, x_true, i_mask = arrays
    ds = TSImputeDataset(x_in[indices], m_hat[indices], x_true[indices],
                         i_mask[indices], indices)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=train,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)


def split_folds(seg_id):
    patients = np.unique(seg_id)
    rng = np.random.default_rng(SEED)
    shuffled = patients.copy()
    rng.shuffle(shuffled)
    return patients, np.array_split(shuffled, N_SPLITS)


def train_val_patients(trainval, fold):
    pats = np.asarray(trainval).copy()
    rng = np.random.default_rng(SEED + fold)
    rng.shuffle(pats)
    n_val = min(max(1, int(round(len(pats) * VAL_WITHIN_TRAIN_P))), len(pats) - 1)
    return pats[n_val:], pats[:n_val]


def point_metrics(sdf):
    yt = sdf['y_true'].to_numpy(np.float64)
    yp = sdf['y_pred'].to_numpy(np.float64)
    valid = np.isfinite(yt) & np.isfinite(yp)
    ae = np.abs(yp[valid] - yt[valid])
    nz = np.abs(yt[valid]) > EPS
    return dict(mae=float(ae.mean()) if len(ae) else np.nan,
                rmse=float(np.sqrt(np.mean(ae**2))) if len(ae) else np.nan,
                mape_pct=float((100*ae[nz]/np.abs(yt[valid][nz])).mean()) if nz.any() else np.nan,
                n_points=int(len(ae)), n_mape_points=int(nz.sum()))


def gap_lengths(target_mask_1d):
    m = np.asarray(target_mask_1d) > 0.5
    padded = np.r_[False, m, False]
    diff = np.diff(padded.astype(np.int8))
    lengths = np.zeros(len(m), dtype=np.int32)
    for start, end in zip(np.flatnonzero(diff == 1), np.flatnonzero(diff == -1)):
        lengths[start:end] = end-start
    return lengths


def gap_region(g):
    if 1 <= g < 12: return REGIONS[0]
    if 12 <= g < 120: return REGIONS[1]
    if 120 <= g < 240: return REGIONS[2]
    if 240 <= g <= 360: return REGIONS[3]
    return None


def to_device(batch):
    xb, mb, yb, ib, ids = batch
    return (xb.to(DEVICE, non_blocking=True),
            mb.to(DEVICE, non_blocking=True),
            yb.to(DEVICE, non_blocking=True),
            ib.to(DEVICE, non_blocking=True), ids)


def run_training_epoch(model, loader, optimizer):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    loss_total = nll_total = kl_total = 0.0
    n_samples = 0
    n_batches = len(loader)
    for index, batch in enumerate(loader):
        xb, mb, yb, ib, _ = to_device(batch)
        output = model(xb, mb, sample=True)
        loss, logs = gpvae_original_elbo_loss(output, yb, mb, ib)
        if not torch.isfinite(loss).item():
            raise FloatingPointError('Nonfinite GP-VAE ELBO. Inspect KL, inputs and learning rate.')
        start = (index // GRAD_ACCUM_STEPS) * GRAD_ACCUM_STEPS
        group_end = min(start + GRAD_ACCUM_STEPS, n_batches)
        group_n = min(len(loader.dataset), group_end*BATCH_SIZE) - start*BATCH_SIZE
        batch_n = xb.shape[0]
        (loss*(batch_n/group_n)).backward()
        loss_total += float(loss.detach()) * batch_n
        nll_total += float(logs['nll']) * batch_n
        kl_total += float(logs['kl_loss']) * batch_n
        n_samples += batch_n
        if ((index+1) % GRAD_ACCUM_STEPS == 0) or (index+1 == n_batches):
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD_NORM)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
    return dict(elbo=loss_total/n_samples, nll=nll_total/n_samples,
                kl=kl_total/n_samples)


@torch.no_grad()
def run_eval_epoch(model, loader):
    """GT is consulted only for evaluation; never fed as hidden model inputs."""
    model.eval()
    loss_sum = nll_sum = kl_sum = 0.0
    point_abs = point_squared = 0.0
    n_samples = n_points = 0
    for batch in loader:
        xb, mb, yb, ib, _ = to_device(batch)
        outputs = model(xb, mb, sample=False)
        loss, logs = gpvae_original_elbo_loss(outputs, yb, mb, ib)
        if not torch.isfinite(loss).item():
            raise FloatingPointError('Nonfinite GP-VAE validation ELBO')
        bn = xb.shape[0]
        loss_sum += float(loss)*bn
        nll_sum += float(logs['nll'])*bn
        kl_sum += float(logs['kl_loss'])*bn
        n_samples += bn
        keep = (ib > 0.5) & (mb <= 0.5) & torch.isfinite(yb)
        err = outputs['x_hat'][keep]-yb[keep]
        if err.numel():
            e = err.float()
            point_abs += float(e.abs().sum())
            point_squared += float(e.square().sum())
            n_points += e.numel()
    if n_points == 0:
        raise ValueError('No finite artificial-gap GT in validation/test split')
    return dict(elbo=loss_sum/n_samples, nll=nll_sum/n_samples,
                kl=kl_sum/n_samples, imputation_mae=point_abs/n_points,
                imputation_rmse=(point_squared/n_points)**0.5,
                n_imputation_points=n_points)


@torch.no_grad()
def collect_pointwise_predictions(model, loader, seg_id, filenames, target_names, fold, split):
    model.eval()
    rows = []
    for batch in loader:
        xb, mb, yb, ib, global_indices = to_device(batch)
        outputs = model(xb, mb, sample=False)
        preds = outputs['x_c'].float().cpu().numpy()
        truths = yb.cpu().numpy()
        i_np = ib.cpu().numpy()
        m_np = mb.cpu().numpy()
        for b, global_idx in enumerate(global_indices.tolist()):
            for d, channel in enumerate(target_names):
                lengths = gap_lengths(i_np[b, :, d])
                for t in np.flatnonzero((i_np[b, :, d] > .5) & (m_np[b, :, d] <= .5)):
                    truth, pred = float(truths[b, t, d]), float(preds[b, t, d])
                    if not (np.isfinite(truth) and np.isfinite(pred)):
                        continue
                    g = int(lengths[t])
                    rows.append(dict(fold=int(fold), split=split,
                                     segment_idx=int(global_idx),
                                     patient_id=seg_id[global_idx].item() if hasattr(seg_id[global_idx], 'item') else seg_id[global_idx],
                                     filename=str(filenames[global_idx]), time_idx=int(t),
                                     target=channel, model=MODEL_NAME,
                                     y_true=truth, y_pred=pred,
                                     abs_err=abs(truth-pred), gap_len=g,
                                     gap_region=gap_region(g)))
    return pd.DataFrame(rows)


def save_csv(df, name):
    path = os.path.join(OUT_DIR, name)
    df.to_csv(path, index=False)
    return path


def grouped_metrics(df, cols, prefix):
    if df.empty:
        return pd.DataFrame()
    output = []
    for values, sub in df.groupby(cols, observed=True):
        values = values if isinstance(values, tuple) else (values,)
        output.append({**dict(zip(cols, values)),
                       **{f'{prefix}_{k}': v for k, v in point_metrics(sub).items()}})
    return pd.DataFrame(output)


def save_all_summaries(fold_rows, test_frames, val_frames, split_rows):
    fold_df = pd.DataFrame(fold_rows)
    test_df = pd.concat(test_frames, ignore_index=True)
    val_df = pd.concat(val_frames, ignore_index=True)
    save_csv(fold_df, 'cv5_fold_results.csv')
    save_csv(test_df, 'cv5_test_predictions_pooled.csv')
    save_csv(val_df, 'cv5_validation_predictions_pooled.csv')
    save_csv(pd.DataFrame(split_rows), 'cv5_patient_split_summary.csv')

    for subset, prefix in [(test_df, 'test'), (val_df, 'validation')]:
        gap_df = subset.loc[subset['gap_region'].notna()].copy()
        if not gap_df.empty:
            gap_df['gap_region'] = pd.Categorical(gap_df['gap_region'], REGIONS, ordered=True)
        save_csv(gap_df, f'cv5_{prefix}_predictions_pooled_with_gap_region.csv')
        save_csv(grouped_metrics(subset, ['target', 'model'], 'pooled_'+prefix),
                 f'cv5_pooled_{prefix}_metrics.csv')
        save_csv(grouped_metrics(gap_df, ['target', 'model', 'gap_region'], 'pooled_'+prefix),
                 f'cv5_pooled_{prefix}_metrics_by_gap_region.csv')
        fold_gap = grouped_metrics(gap_df, ['fold','target','model','gap_region'], prefix)
        save_csv(fold_gap, f'cv5_{prefix}_fold_metrics_by_gap_region.csv')
        if prefix == 'test':
            save_csv(fold_gap, 'cv5_fold_metrics_by_gap_region.csv')
        if not fold_gap.empty:
            gap_summary = fold_gap.groupby(['target','model','gap_region'], observed=True, as_index=False).agg(
                mae_mean=(f'{prefix}_mae', 'mean'), mae_std=(f'{prefix}_mae','std'),
                rmse_mean=(f'{prefix}_rmse','mean'), rmse_std=(f'{prefix}_rmse','std'),
                mape_mean_pct=(f'{prefix}_mape_pct','mean'), mape_std_pct=(f'{prefix}_mape_pct','std'),
                n_folds=('fold','nunique'), total_points=(f'{prefix}_n_points', 'sum'))
        else:
            gap_summary = pd.DataFrame()
        save_csv(gap_summary, f'cv5_{prefix}_summary_mean_std_by_gap_region.csv')
        if prefix == 'test':
            save_csv(gap_summary, 'cv5_summary_mean_std_by_gap_region.csv')

    fold_summary = fold_df.groupby(['target','model'], as_index=False).agg(
        val_imputation_mae_mean=('val_imputation_mae','mean'),
        val_imputation_mae_std=('val_imputation_mae','std'),
        val_elbo_mean=('val_elbo','mean'), val_elbo_std=('val_elbo','std'),
        test_mae_mean=('test_mae','mean'), test_mae_std=('test_mae','std'),
        test_rmse_mean=('test_rmse','mean'), test_rmse_std=('test_rmse','std'),
        test_mape_mean_pct=('test_mape_pct','mean'), test_mape_std_pct=('test_mape_pct','std'),
        n_folds=('fold','nunique'), total_points=('n_points','sum'))
    save_csv(fold_summary, 'cv5_summary_mean_std.csv')
    print('\nFive-fold summary (completed folds):\n', fold_summary.to_string(index=False))
    print('\nPooled held-out test metrics:\n',
          grouped_metrics(test_df,['target','model'],'pooled_test').to_string(index=False))


def main():
    if 'GPVAEImputer' not in globals() or 'gpvae_original_elbo_loss' not in globals():
        raise NameError('Run the new GPVAEImputer model cell first, including gpvae_original_elbo_loss. '
                        'For Python files in Colab use: %run -i /path/to/train_gpvae_original_elbo_5fold.py')
    seed_all(SEED)
    os.makedirs(CKPT_DIR, exist_ok=True)
    x_in, m_hat, x_true, i_mask, seg_id, filenames, target_names = load_data()
    arrays = (x_in, m_hat, x_true, i_mask)
    patients, patient_folds = split_folds(seg_id)
    print('DEVICE:',DEVICE,'DATA:',x_in.shape,'PATIENTS:',len(patients))
    print('Training objective: observed Gaussian NLL + beta * GP-prior KL')
    print('Checkpoint selection: validation artificial-gap MAE (NOT the training loss)')
    print('Model:',MODEL_KWARGS)
    fold_rows, test_frames, val_frames, split_rows = [], [], [], []
    for fold, test_pats in enumerate(patient_folds, 1):
        test_set = set(test_pats.tolist())
        trainval_pats = np.array([p for p in patients if p not in test_set])
        train_pats, val_pats = train_val_patients(trainval_pats, fold)
        train_idx = np.flatnonzero(np.isin(seg_id,train_pats))
        val_idx = np.flatnonzero(np.isin(seg_id,val_pats))
        test_idx = np.flatnonzero(np.isin(seg_id,test_pats))
        if min(map(len,(train_idx,val_idx,test_idx))) == 0:
            raise ValueError(f'Fold {fold}: empty train, val, or test partition')
        print('\n'+'='*80)
        print(f'FOLD {fold}/{N_SPLITS} | patients train/val/test: '
              f'{len(train_pats)}/{len(val_pats)}/{len(test_pats)} | segments '
              f'{len(train_idx)}/{len(val_idx)}/{len(test_idx)}')
        split_rows.append(dict(
            fold=fold,
            train_patients=','.join(map(str, sorted(train_pats.tolist()))),
            val_patients=','.join(map(str, sorted(val_pats.tolist()))),
            test_patients=','.join(map(str, sorted(test_pats.tolist()))),
            n_train_patients=len(train_pats),n_val_patients=len(val_pats),
            n_test_patients=len(test_pats),n_train_segments=len(train_idx),
            n_val_segments=len(val_idx),n_test_segments=len(test_idx)))
        train_loader = make_loader(arrays, train_idx, train=True)
        val_loader = make_loader(arrays, val_idx)
        test_loader = make_loader(arrays, test_idx)
        model = GPVAEImputer(**MODEL_KWARGS).to(DEVICE)
        if MODEL_KWARGS['normalization'] == 'fold':
            obs_values = np.where(m_hat[train_idx] > .5, x_in[train_idx], np.nan)
            train_mu = np.nanmean(obs_values, axis=(0,1))
            train_sd = np.nanstd(obs_values, axis=(0,1))
            if not np.isfinite(train_mu).all():
                raise ValueError('Nonfinite train-only fold normalization means')
            model.set_standardization(train_mu, train_sd)
        optimizer = torch.optim.Adam(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,mode='min',factor=0.5,patience=3)
        ckpt = os.path.join(CKPT_DIR,f'fold{fold:02d}_best.pt')
        best_val, best_epoch, no_improve = float('inf'),None,0
        history=[]
        for epoch in range(1,EPOCHS+1):
            tr = run_training_epoch(model,train_loader,optimizer)
            va = run_eval_epoch(model,val_loader)
            scheduler.step(va['imputation_mae'])
            lr_now = optimizer.param_groups[0]['lr']
            history.append(dict(fold=fold,epoch=epoch,lr=lr_now,
                                train_elbo=tr['elbo'],train_nll=tr['nll'],train_kl=tr['kl'],
                                val_elbo=va['elbo'],val_nll=va['nll'],val_kl=va['kl'],
                                val_imputation_mae=va['imputation_mae'],
                                val_imputation_rmse=va['imputation_rmse'],
                                val_n_target_points=va['n_imputation_points']))
            save_csv(pd.DataFrame(history),f'fold{fold:02d}_training_history.csv')
            print(f'Epoch {epoch:03d} lr={lr_now:.2e} | '
                  f'train ELBO={tr["elbo"]:.5f} (NLL={tr["nll"]:.5f}, KL={tr["kl"]:.5f}) | '
                  f'val ELBO={va["elbo"]:.5f}, val imputation MAE={va["imputation_mae"]:.5f}, '
                  f'RMSE={va["imputation_rmse"]:.5f}')
            if va['imputation_mae'] < best_val:
                best_val, best_epoch, no_improve = va['imputation_mae'],epoch,0
                torch.save(model.state_dict(),ckpt)
                print('  Saved best checkpoint:',ckpt)
            else:
                no_improve+=1
                if no_improve>=EARLY_STOPPING_PATIENCE:
                    print('  Early stopping at epoch',epoch)
                    break
        model.load_state_dict(torch.load(ckpt,map_location=DEVICE,weights_only=True))
        model.eval()
        val_metrics=run_eval_epoch(model,val_loader)
        test_metrics=run_eval_epoch(model,test_loader)
        print('Best epoch',best_epoch,'| val imputation MAE',val_metrics['imputation_mae'])
        val_predictions=collect_pointwise_predictions(
            model,val_loader,seg_id,filenames,target_names,fold,'val')
        test_predictions=collect_pointwise_predictions(
            model,test_loader,seg_id,filenames,target_names,fold,'test')
        if test_predictions.empty or val_predictions.empty:
            raise ValueError(f'Fold {fold}: missing finite predictions')
        save_csv(val_predictions,f'fold{fold:02d}_validation_predictions.csv')
        save_csv(test_predictions,f'fold{fold:02d}_test_predictions.csv')
        test_frames.append(test_predictions)
        val_frames.append(val_predictions)
        val_by_target = {channel: point_metrics(sub)
                         for channel, sub in val_predictions.groupby('target')}
        for channel, df in test_predictions.groupby('target'):
            m=point_metrics(df)
            vm=val_by_target[channel]
            fold_rows.append(dict(fold=fold,target=channel,model=MODEL_NAME,
                                  best_epoch=best_epoch,
                                  val_elbo=val_metrics['elbo'],val_nll=val_metrics['nll'],
                                  val_kl=val_metrics['kl'],
                                  val_imputation_mae=vm['mae'],
                                  val_imputation_rmse=vm['rmse'],
                                  val_overall_imputation_mae=val_metrics['imputation_mae'],
                                  test_elbo=test_metrics['elbo'],
                                  test_mae=m['mae'],test_rmse=m['rmse'],
                                  test_mape_pct=m['mape_pct'],n_points=m['n_points'],
                                  n_train_patients=len(train_pats),n_val_patients=len(val_pats),
                                  n_test_patients=len(test_pats),n_train_segments=len(train_idx),
                                  n_val_segments=len(val_idx),n_test_segments=len(test_idx)))
            print(f'  Held-out TEST {channel}: MAE={m["mae"]:.4f} '
                  f'RMSE={m["rmse"]:.4f} MAPE={m["mape_pct"]:.2f}%')
        save_all_summaries(fold_rows,test_frames,val_frames,split_rows)
        del model, optimizer, scheduler
        if DEVICE.type=='cuda': torch.cuda.empty_cache()
    print('\nAll folds completed. Saved under:',OUT_DIR)

if __name__ == '__main__':
    main()

NPZ channel names: None
Benchmark channel names (user-specified): ['ScO2', 'MAP']
DEVICE: cuda DATA: (3817, 1080, 2) PATIENTS: 64
Training objective: observed Gaussian NLL + beta * GP-prior KL
Checkpoint selection: validation artificial-gap MAE (NOT the training loss)
Model: {'seq_len': 1080, 'data_dim': 2, 'latent_dim': 35, 'encoder_sizes': (128, 128), 'decoder_sizes': (256, 256), 'window_size': 24, 'kernel': 'cauchy', 'gp_length_scale': 7.0, 'gp_sigma': 1.005, 'gp_jitter': 0.001, 'kernel_scales': 1, 'beta': 0.2, 'posterior': 'banded', 'normalization': 'observed'}

FOLD 1/5 | patients train/val/test: 46/5/13 | segments 2814/256/747


/usr/local/lib/python3.13/dist-packages/torch/nn/modules/conv.py:370: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1024.)
  return F.conv1d(


Epoch 001 lr=1.00e-03 | train ELBO=1358673.29771 (NLL=2586.03146, KL=6780436.17058) | val ELBO=1313026.75000, val imputation MAE=2.64222, RMSE=3.98303
  Saved best checkpoint: /content/drive/MyDrive/aim3/GPVAE_OriginalELBO_Banded_5fold_1e-3_256/ckpts/fold01_best.pt
Epoch 002 lr=1.00e-03 | train ELBO=1291376.18008 (NLL=2522.27801, KL=6444269.34648) | val ELBO=1241904.62500, val imputation MAE=2.59252, RMSE=3.90748
  Saved best checkpoint: /content/drive/MyDrive/aim3/GPVAE_OriginalELBO_Banded_5fold_1e-3_256/ckpts/fold01_best.pt
Epoch 003 lr=1.00e-03 | train ELBO=1215145.64765 (NLL=2423.92883, KL=6063608.54016) | val ELBO=1155657.87500, val imputation MAE=2.47236, RMSE=3.73957
  Saved best checkpoint: /content/drive/MyDrive/aim3/GPVAE_OriginalELBO_Banded_5fold_1e-3_256/ckpts/fold01_best.pt
Epoch 004 lr=1.00e-03 | train ELBO=1123704.08404 (NLL=2285.39676, KL=5607093.25693) | val ELBO=1056486.00000, val imputation MAE=2.28781, RMSE=3.50382
  Saved best checkpoint: /content/drive/MyDrive/aim

### Validation performance

In [ ]:
"""Recompute five-fold validation metrics from trained ORIGINAL-OBJECTIVE GP-VAE checkpoints.

Run AFTER defining the updated GPVAEImputer and gpvae_original_elbo_loss from:
    gpvae_original_objective_pytorch_1080x2.py
The model is NOT redefined here and this script does NOT train or update weights.

Training: observed-data Gaussian negative log likelihood + beta * GP prior KL.
Checkpoint selection in train_gpvae_original_elbo_5fold.py: validation imputation MAE.
This script evaluates both at the five saved validation-patient folds.

IMPORTANT:
- Imputed values are generated using X_in and M_hat ONLY; GT X is evaluation-only.
- For posterior='banded', KL is one-sample Monte Carlo even with sample=False;
  therefore recomputed ELBO/KL can vary slightly, although imputation predictions
  (posterior mean) are deterministic with model.eval().
- Do not use checkpoints from the old MAE-based GP-VAE or C2FFormer.
- Uses the same MODEL_KWARGS and output directory as the updated GP-VAE trainer.
"""

import os
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

# ============================================================================
# Configuration
# ============================================================================

CKPT_DIR = os.path.join(OUT_DIR, "ckpts")
SPLIT_CSV = os.path.join(OUT_DIR, "cv5_patient_split_summary.csv")

SEQ_LEN = 1080
D_IN = 2
N_SPLITS = 5
BATCH_SIZE = 4
NUM_WORKERS = 0
SEED = 1337
EPS = 1e-8
COLLECT_POINTWISE_PREDICTIONS = True

MODEL_KWARGS = dict(
    seq_len=SEQ_LEN,
    data_dim=D_IN,
    latent_dim=35,
    encoder_sizes=(128, 128),
    decoder_sizes=(256, 256),
    window_size=24,
    kernel="cauchy",
    gp_length_scale=7.0,
    gp_sigma=1.005,
    gp_jitter=1e-3,
    kernel_scales=1,
    beta=0.2,
    posterior="banded",
    normalization="observed",
)
MODEL_NAME = "GPVAE_OriginalELBO_" + MODEL_KWARGS["posterior"]
REGIONS = ["<1 min", "[1,10) min", "[10,20) min", "[20,30] min"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = DEVICE.type == "cuda"


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def save_csv(frame, name):
    os.makedirs(OUT_DIR, exist_ok=True)
    path = os.path.join(OUT_DIR, name)
    frame.to_csv(path, index=False)
    print("Saved:", path)
    return path


class ValidationDataset(Dataset):
    def __init__(self, x_in, m_hat, truth, i_mask, indices):
        self.x_in = torch.from_numpy(
            np.nan_to_num(x_in, nan=0., posinf=0., neginf=0.).astype(np.float32)
        )
        self.m_hat = torch.from_numpy(
            np.nan_to_num(m_hat, nan=0., posinf=0., neginf=0.).astype(np.float32)
        )
        # Keep NaNs in GT so naturally missing values are not scored.
        self.truth = torch.from_numpy(np.asarray(truth, dtype=np.float32))
        self.i_mask = torch.from_numpy(
            np.nan_to_num(i_mask, nan=0., posinf=0., neginf=0.).astype(np.float32)
        )
        self.indices = torch.as_tensor(indices, dtype=torch.long)
        assert self.x_in.shape == self.m_hat.shape == self.truth.shape == self.i_mask.shape
        assert len(self.indices) == len(self.x_in)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        return (self.x_in[index], self.m_hat[index], self.truth[index],
                self.i_mask[index], self.indices[index])


def parse_patients(value):
    if pd.isna(value) or not str(value).strip():
        return []
    return [int(float(p.strip())) for p in str(value).split(",") if p.strip()]


def gap_lengths(mask):
    masked = np.asarray(mask) > 0.5
    changes = np.diff(np.r_[False, masked, False].astype(np.int8))
    starts = np.flatnonzero(changes == 1)
    ends = np.flatnonzero(changes == -1)
    result = np.zeros(len(masked), dtype=np.int32)
    for start, end in zip(starts, ends):
        result[start:end] = end - start
    return result


def gap_region(length):
    # Exact five-second sampling boundaries; length 360 is exactly 30 minutes.
    if 1 <= length < 12: return REGIONS[0]
    if 12 <= length < 120: return REGIONS[1]
    if 120 <= length < 240: return REGIONS[2]
    if 240 <= length <= 360: return REGIONS[3]
    return None


def point_metrics(df):
    yt = df["y_true"].to_numpy(dtype=np.float64)
    yp = df["y_pred"].to_numpy(dtype=np.float64)
    finite = np.isfinite(yt) & np.isfinite(yp)
    ae = np.abs(yt[finite] - yp[finite])
    nz = np.abs(yt[finite]) > EPS
    return dict(
        mae=float(ae.mean()) if len(ae) else np.nan,
        rmse=float(np.sqrt(np.mean(ae**2))) if len(ae) else np.nan,
        mape_pct=float((100*ae[nz]/np.abs(yt[finite][nz])).mean()) if nz.any() else np.nan,
        n_points=int(finite.sum()),
        n_mape=int(nz.sum()),
    )


def grouped_metrics(df, columns):
    result = []
    for keys, part in df.groupby(columns, observed=True):
        keys = keys if isinstance(keys, tuple) else (keys,)
        result.append({**dict(zip(columns, keys)), **point_metrics(part)})
    return pd.DataFrame(result)


def main():
    if "GPVAEImputer" not in globals() or "gpvae_original_elbo_loss" not in globals():
        raise NameError(
            "Define GPVAEImputer and gpvae_original_elbo_loss first; "
            "in Colab use `%run -i /path/to/recompute_gpvae_original_elbo_5fold_validation.py`."
        )
    if not os.path.isfile(SPLIT_CSV):
        raise FileNotFoundError(f"No GP-VAE training split CSV: {SPLIT_CSV}")

    seed_all(SEED)
    with np.load(NPZ_PATH, allow_pickle=True) as data:
        x_in = data["X_in"]
        m_hat = data["M_hat"]
        truth = data["X"]
        i_mask = data["I"]
        patient_ids = data["segment_ids"]
        filenames = (np.asarray([str(v) for v in data["filenames"]])
                     if "filenames" in data.files else
                     np.array([f"segment_{i}" for i in range(len(x_in))]))
        recorded_names = (list(map(str, data["target_columns"]))
                          if "target_columns" in data.files else None)

    if x_in.ndim != 3 or x_in.shape[1:] != (SEQ_LEN, D_IN):
        raise ValueError(f"Expected (N,{SEQ_LEN},{D_IN}), found {x_in.shape}")
    if not (x_in.shape == m_hat.shape == truth.shape == i_mask.shape):
        raise ValueError("Input, GT and mask shapes do not match")
    if len(patient_ids) != len(x_in):
        raise ValueError("Patient IDs and segment counts differ")
    if np.any((m_hat > .5) & (i_mask > .5)):
        raise ValueError("Observed M_hat and artificial-gap I mask overlap")
    if not np.isfinite(x_in[m_hat > .5]).all():
        raise ValueError("Observed values in X_in must be finite")

    channels = ["ScO2", "MAP"]
    print("NPZ target names:", recorded_names, "| Evaluated channels:", channels)
    splits = pd.read_csv(SPLIT_CSV).sort_values("fold")
    if len(splits) != N_SPLITS or splits["fold"].nunique() != N_SPLITS:
        raise ValueError(f"Expected exactly {N_SPLITS} patient-level folds")
    print("Device:", DEVICE, "| Model:", MODEL_NAME, "| dataset:", x_in.shape)
    print("Training objective: Gaussian NLL + beta * GP-prior KL")
    print("Checkpoint selection: validation artificial-gap MAE, not ELBO")
    print("Banded posterior KL is Monte Carlo; recomputed ELBO can vary slightly.")

    fold_rows, prediction_frames = [], []
    for _, split in splits.iterrows():
        fold = int(split["fold"])
        val_patients = parse_patients(split["val_patients"])
        test_patients = parse_patients(split["test_patients"])
        train_patients = parse_patients(split["train_patients"])
        if set(val_patients) & (set(train_patients) | set(test_patients)):
            raise ValueError(f"Fold {fold}: patient leakage in recorded train/val/test split")
        indices = np.flatnonzero(np.isin(patient_ids, val_patients))
        if not len(indices):
            raise ValueError(f"Fold {fold}: no validation segments")
        if ("n_val_segments" in splits.columns and pd.notna(split["n_val_segments"])
                and len(indices) != int(split["n_val_segments"])):
            raise ValueError(f"Fold {fold}: NPZ validation segments differ from saved split")

        dataset = ValidationDataset(x_in[indices], m_hat[indices], truth[indices],
                                    i_mask[indices], indices)
        loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
        model = GPVAEImputer(**MODEL_KWARGS).to(DEVICE)
        checkpoint = os.path.join(CKPT_DIR, f"fold{fold:02d}_best.pt")
        if not os.path.isfile(checkpoint):
            raise FileNotFoundError(checkpoint)
        state = torch.load(checkpoint, map_location=DEVICE, weights_only=True)
        model.load_state_dict(state, strict=True)
        model.eval()

        sample_count = target_count = 0
        elbo_sum = nll_sum = kl_sum = abs_error_sum = square_error_sum = 0.0
        pred_rows = []
        with torch.no_grad():
            for xb, mb, yb, ib, idxb in loader:
                xb = xb.to(DEVICE, non_blocking=True)
                mb = mb.to(DEVICE, non_blocking=True)
                yb = yb.to(DEVICE, non_blocking=True)
                ib = ib.to(DEVICE, non_blocking=True)
                outputs = model(xb, mb, sample=False)
                total, logs = gpvae_original_elbo_loss(outputs, yb, mb, ib)
                values = (total, logs["nll"], logs["kl_loss"])
                if any(not torch.isfinite(v).item() for v in values):
                    raise FloatingPointError(f"Fold {fold}: nonfinite validation ELBO/NLL/KL")
                bsize = len(xb)
                elbo_sum += float(total) * bsize
                nll_sum += float(logs["nll"]) * bsize
                kl_sum += float(logs["kl_loss"]) * bsize
                sample_count += bsize
                mask = (ib > .5) & (mb <= .5) & torch.isfinite(yb)
                errors = (outputs["x_hat"] - yb)[mask]
                if errors.numel():
                    errors = errors.float()
                    abs_error_sum += float(errors.abs().sum())
                    square_error_sum += float(errors.square().sum())
                    target_count += errors.numel()

                if COLLECT_POINTWISE_PREDICTIONS:
                    preds = outputs["x_c"].float().cpu().numpy()
                    y_np = yb.cpu().numpy()
                    m_np = mb.cpu().numpy()
                    i_np = ib.cpu().numpy()
                    for b, global_idx in enumerate(idxb.tolist()):
                        for d, channel in enumerate(channels):
                            lengths = gap_lengths(i_np[b, :, d])
                            times = np.flatnonzero((i_np[b, :, d] > .5) &
                                                   (m_np[b, :, d] <= .5))
                            for t in times:
                                actual = float(y_np[b, t, d])
                                predicted = float(preds[b, t, d])
                                if not (np.isfinite(actual) and np.isfinite(predicted)):
                                    continue
                                length = int(lengths[t])
                                pred_rows.append(dict(
                                    fold=fold, split="val", segment_idx=int(global_idx),
                                    patient_id=int(patient_ids[global_idx]),
                                    filename=str(filenames[global_idx]), time_idx=int(t),
                                    target=channel, model=MODEL_NAME,
                                    y_true=actual, y_pred=predicted,
                                    abs_err=abs(actual - predicted),
                                    gap_len=length, gap_region=gap_region(length)
                                ))

        if target_count == 0:
            raise ValueError(f"Fold {fold}: no finite artificial-gap validation GT")
        metrics = dict(
            fold=fold, model=MODEL_NAME,
            val_elbo=elbo_sum/sample_count,
            val_nll=nll_sum/sample_count,
            val_kl=kl_sum/sample_count,
            val_imputation_mae=abs_error_sum/target_count,
            val_imputation_rmse=np.sqrt(square_error_sum/target_count),
            n_val_patients=len(val_patients), n_val_segments=len(indices),
            n_target_points=target_count,
        )
        fold_rows.append(metrics)
        print(f"FOLD {fold}/{N_SPLITS}: val imputation MAE={metrics['val_imputation_mae']:.5f}, "
              f"RMSE={metrics['val_imputation_rmse']:.5f} | "
              f"ELBO objective={metrics['val_elbo']:.5f}, "
              f"NLL={metrics['val_nll']:.5f}, KL={metrics['val_kl']:.5f}")
        if COLLECT_POINTWISE_PREDICTIONS:
            fold_df = pd.DataFrame(pred_rows)
            if fold_df.empty:
                raise ValueError(f"Fold {fold}: missing finite pointwise predictions")
            save_csv(fold_df, f"fold{fold:02d}_validation_predictions_recomputed.csv")
            prediction_frames.append(fold_df)

        del model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    losses = pd.DataFrame(fold_rows).sort_values("fold")
    save_csv(losses, "cv5_validation_losses_recomputed.csv")
    summary = pd.DataFrame([dict(
        model=MODEL_NAME,
        val_elbo_mean=losses["val_elbo"].mean(),
        val_elbo_std=losses["val_elbo"].std(ddof=1),
        val_nll_mean=losses["val_nll"].mean(),
        val_nll_std=losses["val_nll"].std(ddof=1),
        val_kl_mean=losses["val_kl"].mean(),
        val_kl_std=losses["val_kl"].std(ddof=1),
        val_imputation_mae_mean=losses["val_imputation_mae"].mean(),
        val_imputation_mae_std=losses["val_imputation_mae"].std(ddof=1),
        val_imputation_rmse_mean=losses["val_imputation_rmse"].mean(),
        val_imputation_rmse_std=losses["val_imputation_rmse"].std(ddof=1),
        n_folds=len(losses),
    )])
    save_csv(summary, "cv5_validation_loss_summary_mean_std_recomputed.csv")
    print("\nFIVE-FOLD VALIDATION OVERALL MEAN ± SAMPLE SD:\n", summary.to_string(index=False))

    if not COLLECT_POINTWISE_PREDICTIONS:
        return
    pred_df = pd.concat(prediction_frames, ignore_index=True)
    save_csv(pred_df, "cv5_validation_predictions_recomputed_pooled.csv")
    gap_df = pred_df.loc[pred_df["gap_region"].notna()].copy()
    gap_df["gap_region"] = pd.Categorical(gap_df["gap_region"],
                                           categories=REGIONS, ordered=True)
    save_csv(gap_df, "cv5_validation_predictions_recomputed_with_gap_region.csv")

    fold_metrics = grouped_metrics(pred_df, ["fold", "target", "model"])
    save_csv(fold_metrics, "cv5_validation_fold_metrics_recomputed.csv")
    fold_gap = grouped_metrics(gap_df, ["fold", "target", "model", "gap_region"])
    save_csv(fold_gap, "cv5_validation_fold_metrics_recomputed_by_gap_region.csv")
    pooled = grouped_metrics(pred_df, ["target", "model"])
    save_csv(pooled, "cv5_validation_pooled_metrics_recomputed.csv")
    pooled_gap = grouped_metrics(gap_df, ["target", "model", "gap_region"])
    save_csv(pooled_gap, "cv5_validation_pooled_metrics_recomputed_by_gap_region.csv")

    per_channel = fold_metrics.groupby(["target", "model"], as_index=False).agg(
        mae_mean=("mae", "mean"), mae_std=("mae", "std"),
        rmse_mean=("rmse", "mean"), rmse_std=("rmse", "std"),
        mape_mean_pct=("mape_pct", "mean"),
        mape_std_pct=("mape_pct", "std"),
        n_folds=("fold", "nunique"), total_points=("n_points", "sum"),
    )
    save_csv(per_channel, "cv5_validation_metrics_recomputed_mean_std.csv")
    if not fold_gap.empty:
        gap_summary = fold_gap.groupby(["target", "model", "gap_region"],
                                       observed=True, as_index=False).agg(
            mae_mean=("mae", "mean"), mae_std=("mae", "std"),
            rmse_mean=("rmse", "mean"), rmse_std=("rmse", "std"),
            mape_mean_pct=("mape_pct", "mean"),
            mape_std_pct=("mape_pct", "std"),
            n_folds=("fold", "nunique"), total_points=("n_points", "sum"),
        )
    else:
        gap_summary = pd.DataFrame()
    save_csv(gap_summary, "cv5_validation_metrics_recomputed_mean_std_by_gap_region.csv")
    print("\nRECOMPUTED VALIDATION MAE/RMSE/MAPE BY CHANNEL:\n", per_channel.to_string(index=False))
    print("\nRECOMPUTED VALIDATION METRICS BY GAP LENGTH:\n", gap_summary.to_string(index=False))
    print("\nAll results saved under:", OUT_DIR)


if __name__ == "__main__":
    main()


NPZ target names: None | Evaluated channels: ['ScO2', 'MAP']
Device: cuda | Model: GPVAE_OriginalELBO_banded | dataset: (3817, 1080, 2)
Training objective: Gaussian NLL + beta * GP-prior KL
Checkpoint selection: validation artificial-gap MAE, not ELBO
Banded posterior KL is Monte Carlo; recomputed ELBO can vary slightly.
FOLD 1/5: val imputation MAE=2.10095, RMSE=3.30686 | ELBO objective=853072.05371, NLL=2082.02287, KL=4254950.06641
Saved: /content/drive/MyDrive/aim3/GPVAE_OriginalELBO_Banded_5fold_1e-3_256/fold01_validation_predictions_recomputed.csv
FOLD 2/5: val imputation MAE=1.80563, RMSE=2.94376 | ELBO objective=852243.78383, NLL=2056.89379, KL=4250934.38614
Saved: /content/drive/MyDrive/aim3/GPVAE_OriginalELBO_Banded_5fold_1e-3_256/fold02_validation_predictions_recomputed.csv
FOLD 3/5: val imputation MAE=1.97700, RMSE=4.30458 | ELBO objective=827574.93301, NLL=2045.69788, KL=4127646.10986
Saved: /content/drive/MyDrive/aim3/GPVAE_OriginalELBO_Banded_5fold_1e-3_256/fold03_validat

### Model testing

In [ ]:
"""GP-VAE (original-style ELBO) held-out TEST MAE/MAPE by gap duration.

Reads the five-fold pointwise test predictions saved by
train_gpvae_original_elbo_5fold.py. Does NOT train, load checkpoints,
select hyperparameters, or redefine the GP-VAE model.

Channel 0 = ScO2; channel 1 = MAP (as saved by the GP-VAE CV script).
Sampling = 5 s; four mutually exclusive gap lengths plus ALL.
"""

import os
import numpy as np
import pandas as pd

PRED_CSV = os.path.join(OUT_DIR, "cv5_test_predictions_pooled.csv")
MODEL_NAME = "GPVAE_OriginalELBO_banded"
EPS = 1e-8
REGIONS = ["<1 min", "[1,10) min", "[10,20) min", "[20,30] min", "ALL"]


def calculate_metrics(sdf):
    yt = sdf["y_true"].to_numpy(dtype=np.float64)
    yp = sdf["y_pred"].to_numpy(dtype=np.float64)
    ae = np.abs(yp - yt)
    nonzero = np.abs(yt) > EPS
    return {
        "mae": float(ae.mean()),
        "mape_pct": (
            float(100.0 * np.mean(ae[nonzero] / np.abs(yt[nonzero])))
            if nonzero.any() else np.nan
        ),
        "n_points_total": int(len(yt)),
        "n_points_used_mape": int(nonzero.sum()),
        "n_points_excluded_gt0": int((~nonzero).sum()),
    }


def mean_sd(mean, std):
    if pd.isna(mean):
        return "NA"
    if pd.isna(std):
        return f"{mean:.2f} ± NA"
    return f"{mean:.2f} ± {std:.2f}"


def main(out_dir=OUT_DIR):
    pred_csv = os.path.join(out_dir, "cv5_test_predictions_pooled.csv")
    if not os.path.isfile(pred_csv):
        raise FileNotFoundError(
            f"GP-VAE test predictions not found: {pred_csv}\n"
            "Finish the GP-VAE five-fold training/evaluation first."
        )

    pred_df = pd.read_csv(pred_csv)
    required = {"fold", "target", "y_true", "y_pred", "gap_len"}
    missing = required - set(pred_df.columns)
    if missing:
        raise ValueError(f"Missing columns in {pred_csv}: {sorted(missing)}")

    if "split" in pred_df.columns:
        bad_split = pred_df["split"].astype(str).str.lower() != "test"
        if bad_split.any():
            raise ValueError("The prediction CSV includes rows that are NOT held-out test predictions.")

    if "model" not in pred_df.columns:
        pred_df["model"] = MODEL_NAME

    for col in ("fold", "y_true", "y_pred", "gap_len"):
        pred_df[col] = pd.to_numeric(pred_df[col], errors="coerce")

    pred_df = pred_df.loc[
        np.isfinite(pred_df["fold"])
        & np.isfinite(pred_df["y_true"])
        & np.isfinite(pred_df["y_pred"])
    ].copy()
    if pred_df.empty:
        raise ValueError("No finite GP-VAE test predictions are available.")

    pred_df["fold"] = pred_df["fold"].astype(int)
    folds_found = sorted(pred_df["fold"].unique().tolist())
    if folds_found != [1, 2, 3, 4, 5]:
        raise ValueError(f"Expected all five held-out test folds (1..5), found: {folds_found}")

    target_aliases = {
        "channel_0": "ScO2", "channel_1": "MAP",
        "0": "ScO2", "1": "MAP", "ScO₂": "ScO2",
    }
    pred_df["target"] = pred_df["target"].astype(str).replace(target_aliases)
    found_targets = set(pred_df["target"].unique())
    if found_targets != {"ScO2", "MAP"}:
        raise ValueError(
            "Expected ScO2 (channel 0) and MAP (channel 1) in GP-VAE predictions; "
            f"found {sorted(found_targets)}. Check channel order and labels."
        )
        
    pred_df["gap_region"] = pd.cut(
        pred_df["gap_len"],
        bins=[0, 11, 119, 239, 360],
        labels=REGIONS[:-1],
        right=True,
        include_lowest=False,
    )
    n_outside = int(pred_df["gap_region"].isna().sum())
    if n_outside:
        print(f"WARNING: {n_outside} points have gap_len outside 1..360 or missing. "
              "They remain in ALL but are excluded from the four bins.")

    print("Model(s):", pred_df["model"].unique().tolist())
    print("Channel 0: ScO2; Channel 1: MAP")
    print("Held-out test points by gap category:")
    print(pred_df["gap_region"].value_counts(sort=False).to_string())
    print("ALL held-out test points:", len(pred_df))

    gap_points = pred_df.loc[pred_df["gap_region"].notna()].copy()
    all_points = pred_df.copy()
    all_points["gap_region"] = "ALL"
    eval_df = pd.concat([gap_points, all_points], ignore_index=True)
    eval_df["gap_region"] = pd.Categorical(
        eval_df["gap_region"], categories=REGIONS, ordered=True
    )

    fold_rows = []
    for (fold, target, model, region), sub in eval_df.groupby(
        ["fold", "target", "model", "gap_region"], observed=True
    ):
        fold_rows.append({
            "fold": int(fold), "target": target, "model": model,
            "gap_region": str(region), **calculate_metrics(sub)
        })
    fold_df = pd.DataFrame(fold_rows)
    fold_df["gap_region"] = pd.Categorical(
        fold_df["gap_region"], categories=REGIONS, ordered=True
    )
    fold_df = fold_df.sort_values(
        ["target", "model", "gap_region", "fold"]
    ).reset_index(drop=True)

    summary_df = (
        fold_df.groupby(["target", "model", "gap_region"], observed=True)
        .agg(
            mae_mean=("mae", "mean"),
            mae_std=("mae", "std"), 
            mape_mean_pct=("mape_pct", "mean"),
            mape_std_pct=("mape_pct", "std"),
            n_folds_mae=("mae", "count"),
            n_folds_mape=("mape_pct", "count"),
            total_points=("n_points_total", "sum"),
            total_points_used_mape=("n_points_used_mape", "sum"),
            total_points_excluded_gt0=("n_points_excluded_gt0", "sum"),
        )
        .reset_index()
        .sort_values(["target", "model", "gap_region"])
        .reset_index(drop=True)
    )

    pooled_rows = []
    for (target, model, region), sub in eval_df.groupby(
        ["target", "model", "gap_region"], observed=True
    ):
        pooled_rows.append({
            "target": target, "model": model,
            "gap_region": str(region), **calculate_metrics(sub)
        })
    pooled_df = pd.DataFrame(pooled_rows)
    pooled_df["gap_region"] = pd.Categorical(
        pooled_df["gap_region"], categories=REGIONS, ordered=True
    )
    pooled_df = pooled_df.sort_values(
        ["target", "model", "gap_region"]
    ).reset_index(drop=True)

    paper_rows = []
    for (target, model), sub in summary_df.groupby(["target", "model"], observed=True):
        row = {"target": target, "model": model}
        for region in REGIONS:
            matched = sub.loc[sub["gap_region"] == region]
            if matched.empty:
                row[f"{region} | MAE"] = "NA"
                row[f"{region} | MAPE(%)"] = "NA"
            else:
                r = matched.iloc[0]
                row[f"{region} | MAE"] = mean_sd(r["mae_mean"], r["mae_std"])
                row[f"{region} | MAPE(%)"] = mean_sd(
                    r["mape_mean_pct"], r["mape_std_pct"]
                )
        paper_rows.append(row)
    paper_df = pd.DataFrame(paper_rows)

    os.makedirs(out_dir, exist_ok=True)
    outputs = {
        "cv5_fold_mae_mape_by_gap_region.csv": fold_df,
        "cv5_summary_mean_std_mae_mape_by_gap_region.csv": summary_df,
        "cv5_pooled_test_mae_mape_by_gap_region.csv": pooled_df,
        "gpvae_paper4_mae_mape_table.csv": paper_df,
    }
    for name, frame in outputs.items():
        path = os.path.join(out_dir, name)
        frame.to_csv(path, index=False)
        print("Saved:", path)

    print("\nFOLDWISE TEST MAE + MAPE:\n", fold_df.to_string(index=False))
    print("\nFIVE-FOLD MEAN ± SAMPLE SD:\n", summary_df.to_string(index=False))
    print("\nMANUSCRIPT-READY TABLE:\n", paper_df.to_string(index=False))
    return fold_df, summary_df, pooled_df, paper_df


if __name__ == "__main__":
    main()


Model(s): ['GPVAE_OriginalELBO_banded']
Channel 0: ScO2; Channel 1: MAP
Held-out test points by gap category:
gap_region
<1 min         39503
[1,10) min     52673
[10,20) min    26458
[20,30] min    25794
ALL held-out test points: 144428
Saved: /content/drive/MyDrive/aim3/GPVAE_OriginalELBO_Banded_5fold_1e-3_256/cv5_fold_mae_mape_by_gap_region.csv
Saved: /content/drive/MyDrive/aim3/GPVAE_OriginalELBO_Banded_5fold_1e-3_256/cv5_summary_mean_std_mae_mape_by_gap_region.csv
Saved: /content/drive/MyDrive/aim3/GPVAE_OriginalELBO_Banded_5fold_1e-3_256/cv5_pooled_test_mae_mape_by_gap_region.csv
Saved: /content/drive/MyDrive/aim3/GPVAE_OriginalELBO_Banded_5fold_1e-3_256/gpvae_paper4_mae_mape_table.csv

FOLDWISE TEST MAE + MAPE:
  fold target                     model  gap_region      mae  mape_pct  n_points_total  n_points_used_mape  n_points_excluded_gt0
    1    MAP GPVAE_OriginalELBO_banded      <1 min 1.950062  5.584379            2570                2570                      0
    2    MAP 